# EEG motor imagery classification

Reproducible analysis of EEGBCI hand-versus-foot motor imagery using 15 motor-cortex channels, log-variance power features, logistic regression, and leave-one-recording-run-out validation.

In [ ]:
import os
os.environ.setdefault("MNE_DATA", os.path.expanduser("~/mne_data"))

import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mne.datasets import eegbci
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneGroupOut, cross_val_score

## Analysis definition

Each subject contributes runs 6, 10, and 14. For each frequency band, the raw run is filtered, events T1/T2 are mapped to hands/feet, and the 1–2 second post-cue epoch is converted to one log-variance feature per selected channel. The classifier is evaluated by holding out one complete recording run at a time.

In [ ]:
RUNS = [6, 10, 14]
MOTOR_CHANNELS = [
    "FC3", "FC1", "FCz", "FC2", "FC4",
    "C3", "C1", "Cz", "C2", "C4",
    "CP3", "CP1", "CPz", "CP2", "CP4"
]
BANDS = {"8 to 13 Hz": (8, 13), "13 to 30 Hz": (13, 30), "8 to 30 Hz": (8, 30)}
CLASSIFIER = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
LOGO = LeaveOneGroupOut()

In [ ]:
def analyze_subject(subject):
    results = {}
    for band_name, (low, high) in BANDS.items():
        X_all, y_all, groups_all = [], [], []
        for run_number, run in enumerate(RUNS):
            files = eegbci.load_data(subject, [run], path=os.environ.get("MNE_DATA"), update_path=False)
            raw_run = mne.io.read_raw_edf(files[0], preload=True, verbose=False)
            eegbci.standardize(raw_run)
            raw_run.pick(MOTOR_CHANNELS)
            raw_run.filter(low, high, verbose=False)
            events_run, _ = mne.events_from_annotations(
                raw_run, event_id={"T1": 1, "T2": 2}, verbose=False
            )
            epochs_run = mne.Epochs(
                raw_run, events_run, event_id={"hands": 1, "feet": 2},
                tmin=1, tmax=2, baseline=None, preload=True, verbose=False
            )
            X_eeg = epochs_run.get_data()
            X_all.append(np.log(np.var(X_eeg, axis=2) + 1e-12))
            y = epochs_run.events[:, -1]
            y_all.append(y)
            groups_all.extend([run_number] * len(y))
        X_all = np.concatenate(X_all)
        y_all = np.concatenate(y_all)
        groups_all = np.array(groups_all)
        scores = cross_val_score(
            CLASSIFIER, X_all, y_all, cv=LOGO, groups=groups_all, scoring="accuracy"
        )
        results[band_name] = scores.mean()
    return results

In [ ]:
all_results = {subject: analyze_subject(subject) for subject in range(1, 11)}
results_df = pd.DataFrame(all_results).T
results_df.index.name = "Subject"
results_df.to_csv("results/results.csv")
results_df

In [ ]:
means = results_df.mean()
stds = results_df.std()
print("Mean accuracy across subjects:")
print(means)
print("\nStandard deviation:")
print(stds)

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(results_df.columns))
for subject, row in results_df.iterrows():
    ax.plot(x, row.values, marker="o", alpha=0.35, linewidth=1)
ax.plot(x, means.values, marker="o", linewidth=4, label="Mean across subjects")
ax.axhline(0.5, linestyle=":", label="Chance level")
ax.set_xticks(x, results_df.columns)
ax.set_ylabel("Classification accuracy")
ax.set_xlabel("EEG frequency band")
ax.set_title("Motor imagery classification across subjects")
ax.set_ylim(0, 1)
ax.legend()
fig.tight_layout()
fig.savefig("figures/frequency_band_accuracy.png", dpi=300, bbox_inches="tight")
plt.show()